# ABSA Model Sweep on Kaggle

Run one section at a time on a Kaggle GPU notebook. Training artifacts are written to `/kaggle/working/NLP_outputs/sweeps`.

Kaggle keeps files under `/kaggle/working` as notebook outputs after the run finishes. The final export cell creates a compact zip containing metrics and predictions.

Optional Google Drive upload is still available if you create these Kaggle Secrets:

- `GOOGLE_DRIVE_SERVICE_ACCOUNT_JSON`
- `GOOGLE_DRIVE_FOLDER_ID`

## Install Dependencies

Kaggle usually already has PyTorch installed. This cell installs the project ML dependencies plus `PyDrive2` for Google Drive upload.

In [ ]:
!pip install -q transformers accelerate datasets seqeval scikit-learn sentencepiece protobuf PyDrive2

## Configure Project and Output Paths

In [ ]:
import os
from datetime import datetime

REPO_URL = "https://github.com/ngocvo2511/NLP.git"
PROJECT_DIR = "/kaggle/working/NLP"
RUN_NAME = datetime.utcnow().strftime("sweep_%Y%m%d_%H%M%S")
OUTPUT_ROOT = f"/kaggle/working/NLP_outputs/sweeps/{RUN_NAME}"

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.environ["OUTPUT_ROOT"] = OUTPUT_ROOT

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}
!python -m src.absa.validate_data --data-dir data
print("Local outputs:", OUTPUT_ROOT)
print("Run name:", RUN_NAME)

## Optional Google Drive Upload Helper

Skip this cell unless you have configured the two Kaggle Secrets above. Kaggle output files are enough for most runs.

In [ ]:
import json
import os
import shutil
from pathlib import Path

from kaggle_secrets import UserSecretsClient
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

user_secrets = UserSecretsClient()
service_account_json = user_secrets.get_secret("GOOGLE_DRIVE_SERVICE_ACCOUNT_JSON")
DRIVE_PARENT_FOLDER_ID = user_secrets.get_secret("GOOGLE_DRIVE_FOLDER_ID")

SERVICE_ACCOUNT_PATH = "/kaggle/working/google_drive_service_account.json"
with open(SERVICE_ACCOUNT_PATH, "w", encoding="utf-8") as f:
    f.write(service_account_json)

settings = {
    "client_config_backend": "service",
    "service_config": {
        "client_json_file_path": SERVICE_ACCOUNT_PATH,
    },
}

gauth = GoogleAuth(settings=settings)
gauth.ServiceAuth()
drive = GoogleDrive(gauth)

def create_drive_folder(title, parent_id):
    folder = drive.CreateFile({
        "title": title,
        "mimeType": "application/vnd.google-apps.folder",
        "parents": [{"id": parent_id}],
    })
    folder.Upload()
    return folder["id"]

def upload_file_to_drive(local_path, parent_id, title=None):
    local_path = str(local_path)
    title = title or os.path.basename(local_path)
    item = drive.CreateFile({"title": title, "parents": [{"id": parent_id}]})
    item.SetContentFile(local_path)
    item.Upload()
    return item["id"]

def zip_outputs(output_root):
    output_root = Path(output_root)
    zip_base = Path("/kaggle/working") / output_root.name
    archive_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(output_root))
    return archive_path

print("Drive upload is ready.")

## XLM-R Base: BIO With Exact-Span Checkpoint Selection

This reruns the strongest current family, but now selects the best checkpoint by exact span F1 instead of token F1.

In [ ]:
!python -m src.absa.train_transformer \
  --model-name FacebookAI/xlm-roberta-base \
  --data-dir data \
  --output-dir "$OUTPUT_ROOT/xlmr-base-bio-exact" \
  --tag-scheme bio \
  --class-weight none \
  --metric-for-best-model exact_span_f1 \
  --epochs 6 \
  --batch-size 8 \
  --learning-rate 2e-5 \
  --warmup-ratio 0.06 \
  --save-total-limit 2 \
  --fp16

!python -m src.absa.predict_transformer --model-dir "$OUTPUT_ROOT/xlmr-base-bio-exact" --input data/dev.jsonl --output "$OUTPUT_ROOT/xlmr_base_bio_exact_dev_predictions.jsonl"
!python -m src.absa.evaluate --gold data/dev.jsonl --pred "$OUTPUT_ROOT/xlmr_base_bio_exact_dev_predictions.jsonl" --json-output "$OUTPUT_ROOT/xlmr_base_bio_exact_dev_metrics.json"

## Optional: mDeBERTa-v3 Base

This model repeatedly failed in this project setup: fp16 instability, NaN grad norm in one run, and zero predicted spans even with conservative hyperparameters. Keep this section skipped unless you specifically want to debug mDeBERTa.

In [ ]:
# Skipped by default because previous runs produced F1 = 0.
# Remove the leading '#' characters below only if you want to debug it.
# !python -m src.absa.train_transformer \
#   --model-name microsoft/mdeberta-v3-base \
#   --data-dir data \
#   --output-dir "$OUTPUT_ROOT/mdeberta-v3-base-bio" \
#   --tag-scheme bio \
#   --class-weight none \
#   --metric-for-best-model exact_span_f1 \
#   --epochs 6 \
#   --batch-size 4 \
#   --gradient-accumulation-steps 2 \
#   --learning-rate 5e-6 \
#   --warmup-ratio 0.1 \
#   --weight-decay 0.0 \
#   --save-total-limit 2

# !python -m src.absa.predict_transformer --model-dir "$OUTPUT_ROOT/mdeberta-v3-base-bio" --input data/dev.jsonl --output "$OUTPUT_ROOT/mdeberta_v3_base_bio_dev_predictions.jsonl"
# !python -m src.absa.evaluate --gold data/dev.jsonl --pred "$OUTPUT_ROOT/mdeberta_v3_base_bio_dev_predictions.jsonl" --json-output "$OUTPUT_ROOT/mdeberta_v3_base_bio_dev_metrics.json"

## Multilingual BERT Cased

Smaller and older than XLM-R, but useful as a simple multilingual baseline.

In [ ]:
!python -m src.absa.train_transformer \
  --model-name google-bert/bert-base-multilingual-cased \
  --data-dir data \
  --output-dir "$OUTPUT_ROOT/mbert-cased-bio" \
  --tag-scheme bio \
  --class-weight none \
  --metric-for-best-model exact_span_f1 \
  --epochs 6 \
  --batch-size 8 \
  --learning-rate 2e-5 \
  --warmup-ratio 0.06 \
  --save-total-limit 2 \
  --fp16

!python -m src.absa.predict_transformer --model-dir "$OUTPUT_ROOT/mbert-cased-bio" --input data/dev.jsonl --output "$OUTPUT_ROOT/mbert_cased_bio_dev_predictions.jsonl"
!python -m src.absa.evaluate --gold data/dev.jsonl --pred "$OUTPUT_ROOT/mbert_cased_bio_dev_predictions.jsonl" --json-output "$OUTPUT_ROOT/mbert_cased_bio_dev_metrics.json"

## XLM-R Large

Run this only if GPU memory allows it. On T4, use small batch size plus gradient accumulation.

In [ ]:
!python -m src.absa.train_transformer \
  --model-name FacebookAI/xlm-roberta-large \
  --data-dir data \
  --output-dir "$OUTPUT_ROOT/xlmr-large-bio" \
  --tag-scheme bio \
  --class-weight none \
  --metric-for-best-model exact_span_f1 \
  --epochs 4 \
  --batch-size 2 \
  --gradient-accumulation-steps 4 \
  --learning-rate 1e-5 \
  --warmup-ratio 0.06 \
  --save-total-limit 1 \
  --fp16

!python -m src.absa.predict_transformer --model-dir "$OUTPUT_ROOT/xlmr-large-bio" --input data/dev.jsonl --output "$OUTPUT_ROOT/xlmr_large_bio_dev_predictions.jsonl" --batch-size 8
!python -m src.absa.evaluate --gold data/dev.jsonl --pred "$OUTPUT_ROOT/xlmr_large_bio_dev_predictions.jsonl" --json-output "$OUTPUT_ROOT/xlmr_large_bio_dev_metrics.json"

## PhoBERT Base v2

PhoBERT's standard tokenizer is slow and does not expose fast offset mappings. This run uses `--tokenizer-alignment word` to preserve original character offsets while expanding labels to PhoBERT subwords.

In [ ]:
!python -m src.absa.train_transformer \
  --model-name vinai/phobert-base-v2 \
  --data-dir data \
  --output-dir "$OUTPUT_ROOT/phobert-base-v2-bio-wordalign" \
  --tag-scheme bio \
  --class-weight none \
  --metric-for-best-model exact_span_f1 \
  --tokenizer-alignment word \
  --epochs 6 \
  --batch-size 8 \
  --learning-rate 2e-5 \
  --warmup-ratio 0.06 \
  --save-total-limit 2

!python -m src.absa.predict_transformer --model-dir "$OUTPUT_ROOT/phobert-base-v2-bio-wordalign" --input data/dev.jsonl --output "$OUTPUT_ROOT/phobert_base_v2_bio_wordalign_dev_predictions.jsonl"
!python -m src.absa.evaluate --gold data/dev.jsonl --pred "$OUTPUT_ROOT/phobert_base_v2_bio_wordalign_dev_predictions.jsonl" --json-output "$OUTPUT_ROOT/phobert_base_v2_bio_wordalign_dev_metrics.json"

## Export Compact Kaggle Output

This creates a zip with metrics and prediction files only. It is small enough to download from Kaggle Output even when checkpoints are large.

In [ ]:
from pathlib import Path
import json
import shutil

compact_dir = Path('/kaggle/working/absa_compact_outputs')
compact_dir.mkdir(parents=True, exist_ok=True)

summary = {}
for path in sorted(Path(OUTPUT_ROOT).glob('*_metrics.json')):
    shutil.copy2(path, compact_dir / path.name)
    with open(path, encoding='utf-8') as f:
        summary[path.name] = json.load(f).get('micro', {})

for path in sorted(Path(OUTPUT_ROOT).glob('*_predictions.jsonl')):
    shutil.copy2(path, compact_dir / path.name)

with open(compact_dir / 'summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

archive_path = shutil.make_archive('/kaggle/working/absa_compact_outputs', 'zip', compact_dir)
print('Compact output:', archive_path)
print(json.dumps(summary, ensure_ascii=False, indent=2))

## Upload Outputs to Google Drive

This creates a Drive subfolder for the run and uploads small result files for quick inspection.

Set `UPLOAD_FULL_ARCHIVE = True` only when your `GOOGLE_DRIVE_FOLDER_ID` points to a Google Workspace Shared Drive folder. Service accounts do not have storage quota in normal personal Drive folders, so large archive uploads can fail with `Service Accounts do not have storage quota`.

In [ ]:
from pathlib import Path

UPLOAD_FULL_ARCHIVE = False

drive_run_folder_id = create_drive_folder(RUN_NAME, DRIVE_PARENT_FOLDER_ID)
archive_id = None
archive_path = None

if UPLOAD_FULL_ARCHIVE:
    archive_path = zip_outputs(OUTPUT_ROOT)
    archive_id = upload_file_to_drive(archive_path, drive_run_folder_id)

metric_ids = {}
for metric_path in sorted(Path(OUTPUT_ROOT).glob("*_metrics.json")):
    metric_ids[metric_path.name] = upload_file_to_drive(metric_path, drive_run_folder_id)

print("Uploaded run folder:", f"https://drive.google.com/drive/folders/{drive_run_folder_id}")
print("Uploaded archive:", archive_path, archive_id)
print("Uploaded metrics:", metric_ids)

## Notes

Kaggle notebooks can lose internet or GPU time during long sweeps. Run one model section at a time and execute the compact export cell after each important run.